In [215]:
import os
from pathlib import Path
from dotenv import load_dotenv
from rich import print
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model


## Calling LLM

In [216]:
load_dotenv()
MODEL_NAME = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")
MODEL_NAME

'openai/gpt-oss-20b'

# Making first tool

In [217]:
@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y

In [218]:
add

StructuredTool(name='add', description='Add two numbers.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001BF40140A40>)

# Weather tool

In [219]:
@tool
def get_weather(location: str) -> str:
    """Get the weather for a given location."""
    weather = {"Melbourne" : "32°C, Sunny", "Sydney" : "28°C, Partly Cloudy", "Brisbane" : "30°C, Sunny"}
    
    return f"The weather in {location} is {weather.get(location, 'city unknown')}."   

We have used decorative @tool over the get_weather function so it is no more function but tool.To call @tool, we don't use tool_name()

In [220]:
get_weather

StructuredTool(name='get_weather', description='Get the weather for a given location.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001BF40140D60>)

Calling individual attribute of the tool

In [221]:
get_weather.description
get_weather.name

'get_weather'

In [222]:
@tool
def lookup_user(user_id: int) -> str:
    """Lookup a user by ID."""
    users = {1: "Alice", 2: "Bob", 3: "Charlie"}
    
    return f"User ID {user_id} is {users.get(user_id, 'unknown')}."

In [223]:
lookup_user

StructuredTool(name='lookup_user', description='Lookup a user by ID.', args_schema=<class 'langchain_core.utils.pydantic.lookup_user'>, func=<function lookup_user at 0x000001BF40140860>)

In [224]:
lookup_user.name
lookup_user.description

'Lookup a user by ID.'

In [225]:
RESUME_PATH = Path.cwd().parent / "resumes"
RESUME_PATH

WindowsPath('c:/Users/61424/Desktop/Chapter AI/Agentic_AI/src/Project 3_tool_calling/resumes')

In [226]:
@tool
def get_resume(name: str) -> str:
    """Read and return the resume of a given person."""
    name = name.strip()
    if not name:
        raise ValueError("Name must not be empty.")
    file_path = RESUME_PATH / f"{name}.txt"

    return file_path.read_text(encoding="utf-8") if file_path.exists() else "resume not found"
       


In [227]:
get_resume

StructuredTool(name='get_resume', description='Read and return the resume of a given person.', args_schema=<class 'langchain_core.utils.pydantic.get_resume'>, func=<function get_resume at 0x000001BF40140E00>)

In [228]:
TOOLS = [add, get_weather, lookup_user, get_resume]

In [229]:
TOOLS
#type(TOOLS)

[StructuredTool(name='add', description='Add two numbers.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001BF40140A40>),
 StructuredTool(name='get_weather', description='Get the weather for a given location.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001BF40140D60>),
 StructuredTool(name='lookup_user', description='Lookup a user by ID.', args_schema=<class 'langchain_core.utils.pydantic.lookup_user'>, func=<function lookup_user at 0x000001BF40140860>),
 StructuredTool(name='get_resume', description='Read and return the resume of a given person.', args_schema=<class 'langchain_core.utils.pydantic.get_resume'>, func=<function get_resume at 0x000001BF40140E00>)]

In [230]:
TOOLS_DATA = {
                "add": add, 
                "get_weather": get_weather, 
                "lookup_user": lookup_user, 
                "get_resume": get_resume
            }


In [231]:
TOOLS_DATA["add"]

StructuredTool(name='add', description='Add two numbers.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000001BF40140A40>)

# Passing all tools to LLM so it can decide which one to call according to the prompt.

In [232]:
llm = init_chat_model(
    MODEL_NAME,
    model_provider="groq",
    temperature=0.0,
)

llm_with_tools =llm.bind_tools(list(TOOLS_DATA.values()))

# Calling a tool using the LLM with tools


In [247]:
prompt = "What is the weather in Melbourne?"

print(f"[bold cyan] User: [/bold cyan] {prompt}")
message = llm_with_tools.invoke([HumanMessage(content=prompt)])
message

 User:  What is the weather in Melbourne?

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_ce47b373-8ab6-4155-b3c2-1f1496a45134', 'function': {'arguments': '{"location":"Melbourne"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 197, 'total_tokens': 230, 'completion_time': 0.035838643, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.013406918, 'prompt_tokens_details': None, 'queue_time': 0.246158041, 'total_time': 0.049245561}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12448-5903-7901-bccd-de7f0f9846aa-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Melbourne'}, 'id': 'fc_ce47b373-8ab6-4155-b3c2-1f1496a45134', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens

In [270]:
tool_name = message.tool_calls[0]["name"]
tool_arguments = message.tool_calls[0]["args"]
tool_name, tool_arguments

('get_weather', {'location': 'Melbourne'})